In [1]:
# Install and import required libraries
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv("twitter_training.csv", engine='python', header=None)
df.columns = ['id', 'entity', 'sentiment', 'tweet']   # ✅ all lowercase from start

# ── Clean & filter ────────────────────────────────────────────────────────────
df.dropna(inplace=True)
df['sentiment'] = df['sentiment'].astype(str).str.lower()
df = df[df['sentiment'].isin(['positive', 'negative'])]
df['tweet'] = df['tweet'].astype(str).str.lower()
df['label'] = df['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)

print("Unique sentiments:", df['sentiment'].unique())
print("Dataset size:", len(df))

if len(df) > 20000:
    df = df.sample(n=100, random_state=42)

# ── Train/Val/Test split ──────────────────────────────────────────────────────
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['tweet'], df['label'], test_size=0.2, random_state=42 
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def encode_data(texts):
    return tokenizer(list(texts), padding=True, truncation=True, max_length=128)

train_encodings = encode_data(train_texts)
val_encodings   = encode_data(val_texts)
test_encodings  = encode_data(test_texts)

# ── Dataset class ─────────────────────────────────────────────────────────────
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = CustomDataset(train_encodings, train_labels)
val_dataset   = CustomDataset(val_encodings,   val_labels)
test_dataset  = CustomDataset(test_encodings,  test_labels)

# ── Model ─────────────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1_score": f1}

# ── Training arguments ────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",           
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    save_strategy="no"
)

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# ── Train & Evaluate ──────────────────────────────────────────────────────────
trainer.train()
print("Evaluation Results:", trainer.evaluate())

# ── Confusion Matrix ──────────────────────────────────────────────────────────
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# ── Experiment 1: Freeze all BERT layers ──────────────────────────────────────
for param in model.bert.parameters():
    param.requires_grad = False

trainer.train()
print("Frozen BERT Performance:", trainer.evaluate())

# ── Experiment 2: Fine-tune last 2 layers only ────────────────────────────────
for name, param in model.bert.named_parameters():
    param.requires_grad = "encoder.layer.10" in name or "encoder.layer.11" in name

trainer.train()
print("Last 2 Layers Performance:", trainer.evaluate())

print("All tasks completed successfully 🚀")

C:\Users\adesh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unique sentiments: ['positive' 'negative']
Dataset size: 43013


C:\Users\adesh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  0%|          | 0/40 [00:00<?, ?it/s]C:\Users\adesh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:666: UserW

{'eval_loss': 0.5957819223403931, 'eval_accuracy': 0.8, 'eval_precision': 0.6, 'eval_recall': 1.0, 'eval_f1_score': 0.75, 'eval_runtime': 1.213, 'eval_samples_per_second': 8.244, 'eval_steps_per_second': 4.122, 'epoch': 1.0}
{'train_runtime': 75.9736, 'train_samples_per_second': 1.053, 'train_steps_per_second': 0.526, 'train_loss': 0.6908449649810791, 'epoch': 1.0}


100%|██████████| 5/5 [00:00<00:00,  6.26it/s]
C:\Users\adesh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


...

Frozen BERT Performance: {'eval_loss': 0.5906529426574707, 'eval_accuracy': 0.8, 'eval_precision': 0.6, 'eval_recall': 1.0, 'eval_f1_score': 0.75, 'eval_runtime': 1.081, 'eval_samples_per_second': 9.251, 'eval_steps_per_second': 4.625, 'epoch': 1.0}


  0%|          | 0/40 [00:00<?, ?it/s]C:\Users\adesh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
100%|██████████| 40/40 [00:27<00:00,  1.43it/s]
C:\Users\adesh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.45426100492477417, 'eval_accuracy': 0.9, 'eval_precision': 1.0, 'eval_recall': 0.6666666666666666, 'eval_f1_score': 0.8, 'eval_runtime': 1.0379, 'eval_samples_per_second': 9.635, 'eval_steps_per_second': 4.817, 'epoch': 1.0}
{'train_runtime': 27.9772, 'train_samples_per_second': 2.859, 'train_steps_per_second': 1.43, 'train_loss': 0.6029782772064209, 'epoch': 1.0}


100%|██████████| 5/5 [00:00<00:00,  6.19it/s]

Last 2 Layers Performance: {'eval_loss': 0.45426100492477417, 'eval_accuracy': 0.9, 'eval_precision': 1.0, 'eval_recall': 0.6666666666666666, 'eval_f1_score': 0.8, 'eval_runtime': 1.0227, 'eval_samples_per_second': 9.778, 'eval_steps_per_second': 4.889, 'epoch': 1.0}
All tasks completed successfully 🚀
